In [1]:
from typing import List
from langchain_core.documents import Document
from rag_modules import data_preparation
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
# from langchain_community.vectorstores import Chroma
from langchain_chroma import Chroma

/opt/conda/envs/cook-rag-1/lib/python3.12/site-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
data_module = data_preparation.DataPreparationModule('../../data/C8/cook/')
data_module.load_documents()
data_module.chunk_documents();

# 混合检索

## 检索器设置

加载两个检索器，分别是稀疏向量的检索器，使用最普通的BM25，稠密向量检索器，即embedding向量检索

### 关键词检索器/BM25

In [6]:
import jieba
# 1. 定义中文分词函数
def jieba_tokenizer(text: str) -> list[str]:
    # jieba.lcut 会把 "番茄炒蛋怎么做" 切分成 ['番茄', '炒蛋', '怎么', '做']
    return jieba.lcut(text)

# 2. 重新初始化 BM25 检索器，并强制使用中文分词
bm25_retriever = BM25Retriever.from_documents(
    documents=data_module.chunks,
    preprocess_func=jieba_tokenizer,  # 关键就在这一行！
    k=5
)

Building prefix dict from the default dictionary ...
Dumping model to file cache /tmp/jieba.cache
Loading model cost 0.604 seconds.
Prefix dict has been built successfully.


### 稠密向量检索器

In [4]:
embed_model = HuggingFaceEmbeddings(
    model='BAAI/bge-small-zh-v1.5', 
    model_kwargs={'device': 'cpu'}, 
    encode_kwargs={'normalize_embeddings': True}
)

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-zh-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
vectorstore = Chroma(
    collection_name='dish', 
    persist_directory='./chroma_vector_index', 
    embedding_function=embed_model, 
    collection_metadata={"hnsw:space": "cosine"},  # 使用余弦相似度，默认欧式距离
    # create_collection_if_not_exists=False, 
)

In [5]:
vectorstore = Chroma(
    collection_name='dishes', 
    persist_directory='./chroma_vector_index', 
    embedding_function=embed_model, 
    collection_metadata={"hnsw:space": "cosine"},  # 使用余弦相似度，默认欧式距离
)

if vectorstore._collection.count() == 0:
    vectorstore = Chroma.from_documents(
        documents=data_module.chunks, 
        embedding=embed_model,
        collection_name='dishes', 
        persist_directory='./chroma_vector_index'
    )
    print("加载失败，创建" + "dishes" + "集合成功")
else:
    print('加载' + "dishes" + "集合成功")

vectorstore_retriever = vectorstore.as_retriever(search_type='similarity', search_kwargs={'k': 8})

加载dishes集合成功


## RRF混合检索

将稀疏向量查询和稠密向量查询结果进行综合计算分数

### 采用封装的接口

In [6]:
from langchain.retrievers import EnsembleRetriever

In [20]:
hybird_retrievers = EnsembleRetriever(
    retrievers=[bm25_retriever, vectorstore_retriever], 
    weights=[0.3, 0.7], 
    c=60, 
)

### 手写hybird_retriever

In [7]:
def _rrf_rerank(vector_results: List[Document], bm25_results: List[Document], weight) -> List[Document]:
    """RRF (Reciprocal Rank Fusion) 重排"""
    
    # RRF融合算法
    rrf_scores = {}
    k = 60  # RRF参数
    
    # 计算向量检索的RRF分数
    for rank, doc in enumerate(vector_results):
        doc_id = id(doc)
        rrf_scores[doc_id] =  rrf_scores.get(doc_id, 0) + weight[1] * 1 / (k + rank + 1)

    # 计算BM25检索的RRF分数
    for rank, doc in enumerate(bm25_results):
        doc_id = id(doc)
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + weight[0] * 1 / (k + rank + 1)

    # 合并所有文档并按RRF分数排序
    all_docs = {id(doc): doc for doc in vector_results + bm25_results}
    sorted_docs = sorted(all_docs.items(),
                        key=lambda x: rrf_scores.get(x[0], 0),
                        reverse=True)

    return [doc for _, doc in sorted_docs]

def hybrid_search(query: str, top_k, weight, bm25_retriever, vector_retriever) -> List[Document]:
    """混合检索 - 结合向量检索和BM25检索，使用RRF重排"""
    # 分别获取向量检索和BM25检索结果
    vector_docs = vector_retriever.get_relevant_documents(query)
    bm25_docs = bm25_retriever.get_relevant_documents(query)

    # 使用RRF重排
    reranked_docs = _rrf_rerank(vector_docs, bm25_docs, weight)
    return reranked_docs[:top_k]

In [8]:
hybrid_search("西红柿炒鸡蛋", 4, [0.4, 0.6], bm25_retriever, vectorstore_retriever)

/tmp/ipykernel_3335/824968533.py:29: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  vector_docs = vector_retriever.get_relevant_documents(query)


[Document(metadata={'category': '素菜', 'source': '../../data/C8/cook/dishes/vegetable_dish/西红柿炒鸡蛋.md', 'doc_type': 'child', 'batch_index': 521, '主标题': '西红柿炒鸡蛋的做法', 'difficulty': '简单', 'chunk_size': 95, 'dish_name': '西红柿炒鸡蛋', 'chunk_index': 0, 'parent_id': 'c289d37051726beb3eecc5eb60c1c579', 'chunk_id': '113a69ae-f135-40bb-a31f-18b74541b356'}, page_content='# 西红柿炒鸡蛋的做法  \n西红柿炒蛋是中国家常几乎最常见的一道菜肴。它的原材料易于搜集，制作步骤也较为简单，所以非常适合新厨师上手，是很多人学习做菜时做的第一道菜。  \n预估烹饪难度：★★'),
 Document(metadata={'二级标题': '操作', 'dish_name': '西葫芦炒鸡蛋', '主标题': '西葫芦炒鸡蛋的做法', 'doc_type': 'child', 'batch_index': 1564, 'parent_id': '58e0bdc58d342b079c8e7718758bda64', 'category': '素菜', 'chunk_index': 3, 'source': '../../data/C8/cook/dishes/vegetable_dish/西葫芦炒鸡蛋/西葫芦炒鸡蛋.md', 'chunk_size': 234, 'difficulty': '简单', 'chunk_id': '44a37e1f-edfe-4c22-a3c6-dfb604bc4851'}, page_content='## 操作  \n- 西红柿洗净，切成小块，备用\n- 西葫芦洗净，切成边长约为 4cm 的菱形，备用\n- 打三个鸡蛋到碗里，打散搅匀，备用\n- 热锅，锅内放入 5ml - 10ml 食用油\n- 倒入鸡蛋，保持翻炒至鸡蛋成固体，用锅铲分成小块后盛到碗里，备用\n- 锅内放入 5ml - 10ml 食用油，倒入西红

# 检索构建/元数据过滤检索

In [6]:
from langchain.retrievers.self_query.base import SelfQueryRetriever
from langchain.chains.query_constructor.base import AttributeInfo
from langchain_community.chat_models.moonshot import MoonshotChat
import os

In [7]:
metadata_field_info = [
    AttributeInfo(
        name='主标题', 
        description='这是菜谱中文本的主标题', 
        type='string'
    ), 
    AttributeInfo(
        name='二级标题', 
        description='这是菜谱中文本的二级标题', 
        type='string'
    ), 
    AttributeInfo(
        name='三级标题', 
        description='这是菜谱中文本的三级标题', 
        type='string'
    ), 
    AttributeInfo(
        name='category', 
        description='这是菜谱中菜的类别', 
        type='string'
    ), 
    AttributeInfo(
        name='dish_name', 
        description='这是菜谱中菜的名字', 
        type='string'
    ), 
    AttributeInfo(
        name='difficulty', 
        description='这是菜谱中该菜品的制作难度', 
        type='string'
    ), 
]

In [8]:
llm = MoonshotChat(
    model="kimi-k2-0905-preview", 
    temperature=0, 
    max_tokens=1024, 
    api_key=os.getenv("MOONSHOT_API_KEY"), 
)

In [15]:
metadata_retriever = SelfQueryRetriever.from_llm(
    llm=llm, 
    vectorstore=vectorstore, 
    document_contents="各种菜的菜谱", 
    metadata_field_info=metadata_field_info, 
    enable_limit=True,  # 允许大模型自己决定返回数量
    verbose=True,  # 在控制台打印出模型到底生成了什么查询
    search_kwargs={"k": 3}  # 控制检索器检索结果数量
)

In [10]:
queries = [
    "辣椒炒肉", 
    "金汤肥牛", 
    "酸菜鱼"
]

In [17]:
for query in queries:
    print(f"\n--- 查询: '{query}' ---")
    results = metadata_retriever.invoke(query)
    if results:
        for doc in results:
            print(doc.metadata['dish_name'])
    else:
        print("未找到")


--- 查询: '辣椒炒肉' ---
辣椒炒肉
小米辣炒肉
尖椒炒牛肉
辣椒炒肉

--- 查询: '金汤肥牛' ---
日式肥牛丼饭
日式肥牛丼饭
咖喱肥牛
日式肥牛丼饭

--- 查询: '酸菜鱼' ---
清蒸鳜鱼
清蒸鳜鱼
香煎翘嘴鱼
猪肉烩酸菜


# 重排序

In [43]:
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain.retrievers.document_compressors import CrossEncoderReranker

In [51]:
import os
os.environ["TQDM_DISABLE"] = "1"

In [53]:
# 初始化重排模型
reranker_model = HuggingFaceCrossEncoder(
    model_name='BAAI/bge-reranker-v2-m3', 
    model_kwargs={
        'device': 'cpu',
        'cache_folder': '/tmp/huggingface_cache',  # 指定缓存目录，避免默认位置磁盘空间不足
    }
)

# 4. 初始化重排器，设置top_n=5表示每个候选文档最多返回5个与查询相关的文档对
reranker = CrossEncoderReranker(
    model=reranker_model,
    top_n=5, 
    verbose=False
)

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

ValidationError: 1 validation error for CrossEncoderReranker
verbose
  Extra inputs are not permitted [type=extra_forbidden, input_value=False, input_type=bool]
    For further information visit https://errors.pydantic.dev/2.12/v/extra_forbidden

In [48]:
query = '辣椒炒肉？'
docs = vectorstore_retriever.invoke(query)
len(docs)

8

In [49]:
reranker_docs = reranker.compress_documents(docs, query)

In [52]:
reranker_docs[-1]

Document(id='8bc1039e-7039-4b93-aea4-95490923629d', metadata={'parent_id': 'e484945890d3b390b72c5bf45e0f8f9d', 'dish_name': '辣椒炒肉', 'chunk_size': 382, 'chunk_id': 'ff1e5494-1916-4076-9211-b2d70f4da5cf', 'doc_type': 'child', 'difficulty': '中等', 'source': '../../data/C8/cook/dishes/meat_dish/辣椒炒肉.md', 'batch_index': 172, '二级标题': '操作', 'category': '荤菜', '主标题': '辣椒炒肉的做法', 'chunk_index': 3}, page_content='## 操作  \n* 将`青椒`洗净，去除`青椒把`以及`青椒籽`，再用`滚刀手法`切好备用。\n* `大蒜`用刀拍一下，再横切成`蒜瓣`，`生姜`切碎成`姜末`。\n* 将`猪瘦肉`切成`肉片`（顺着猪肉的纹理切，即刀和肉的纹理呈水平线，出来的肉片，纹路呈“川”字）。\n* 将切好的`猪肉`洗净，放入空碗，再加入计算好的`生抽`、`蚝油`、`盐`搅拌均匀，腌制 10 分钟。\n* 热锅，不用倒油，把`切好的青椒`放入锅中，大火干煸至虎皮状后，再加 2g`盐`继续翻炒 1 分钟 后捞起。\n* 不用洗锅，大火热锅，加入份数 * 8ml`油`，等待 30s，加入`蒜瓣`、`姜末`翻炒 15s。\n* 加入腌制好的`猪肉`倒入锅内翻炒 2 分钟，再加入干煸过的`青椒`翻炒 1 分钟。\n* 根据个人口味喜好加入`豆豉`，最后加入`酱油`，继续翻炒 30s。\n* 出锅，盛盘。')

# 显示检索到的子块信息

In [17]:
relevant_chunks = vectorstore_retriever.invoke("小酥肉的配料")

In [41]:
chunk_info = []
for chunk in data_module.chunks:
    dish_name = chunk.metadata.get('dish_name', '未知菜品')
    title = []
    for t in ['主标题', '二级标题', '三级标题']:
        if chunk.metadata.get(t):
            title.append(chunk.metadata.get(t))
    chunk_info.append(f"{dish_name}({'-'.join(title)})")

In [39]:
chunk_info = []
for chunk in data_module.chunks:
    dish_name = chunk.metadata.get('dish_name', '未知菜品')
    # 尝试从内容中提取章节标题
    content_preview = chunk.page_content[:100].strip()
    if content_preview.startswith('#'):
        # 如果是标题开头，提取标题（仅取第一行）
        title_end = content_preview.find('\n') if '\n' in content_preview else len(content_preview)
        section_title = content_preview[:title_end].replace('#', '').strip()
        chunk_info.append(f"{dish_name}({section_title})")
    else:
        chunk_info.append(f"{dish_name}(内容片段)")